### Create the Historical Database

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('football_scouting.db')
cursor = conn.cursor()

# Create a history table to track annual valuation changes
cursor.execute('''
    CREATE TABLE IF NOT EXISTS valuation_history (
        player_name TEXT,
        year INTEGER,
        market_value_m REAL
    )
''')

# Sample data: 3 years of valuation for Salah and Isak
history_data = [
    ('Mohamed Salah', 2022, 90.0), ('Mohamed Salah', 2023, 80.0), ('Mohamed Salah', 2024, 65.0),
    ('Alexander Isak', 2022, 40.0), ('Alexander Isak', 2023, 60.0), ('Alexander Isak', 2024, 75.0)
]
cursor.executemany('INSERT INTO valuation_history VALUES (?,?,?)', history_data)
conn.commit()
print("✅ SQL History Table Created!")

✅ SQL History Table Created!


### Calculating Growth Rate

In [5]:
# SQL Query: Calculate Year-over-Year (YoY) Growth
# Formula: ((Current - Previous) / Previous) * 100
query = """
SELECT 
    player_name, 
    year, 
    market_value_m,
    LAG(market_value_m) OVER (PARTITION BY player_name ORDER BY year) AS prev_year_value,
    ROUND(((market_value_m - LAG(market_value_m) OVER (PARTITION BY player_name ORDER BY year)) 
           / LAG(market_value_m) OVER (PARTITION BY player_name ORDER BY year)) * 100, 2) AS yoy_growth_pct
FROM valuation_history
"""

df_trends = pd.read_sql_query(query, conn)
print("📈 Player Valuation Trends (SQL Growth Analysis):")
display(df_trends)

conn.close()

📈 Player Valuation Trends (SQL Growth Analysis):


,player_name,year,market_value_m,prev_year_value,yoy_growth_pct
0,Alexander Isak,2022,40.0,NaN,NaN
1,Alexander Isak,2023,60.0,40.0,50.00
2,Alexander Isak,2024,75.0,60.0,25.00
3,Mohamed Salah,2022,90.0,NaN,NaN
4,Mohamed Salah,2023,80.0,90.0,-11.11
5,Mohamed Salah,2024,65.0,80.0,-18.75
